# FRED Chroma Query

In [10]:
%reload_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import os
from collections import Counter
from datetime import datetime, timezone


sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../..')))

from apps.agentic.core.constants import FRED_COLLECTION_NAME, FRED_DB_NAME
from apps.agentic.core.document_loaders.fred_document_loader import FREDChromaDocumentLoader
from apps.agentic.core.utils import (set_chatgpt_env, set_langsmith_env, set_tavily_env, 
                                     set_github_env)

DB_PATH = str(Path("../../.db").resolve())

set_chatgpt_env(filedir="../../.keys")
set_langsmith_env(filedir="../../.keys")

## Verify Document Counts

In [11]:
FRED_DB_NAME, FRED_COLLECTION_NAME, DB_PATH, os.listdir(DB_PATH)

('fred',
 'fred',
 '/Users/troy/Develop/gly.fish/yada/.db',
 ['.DS_Store', 'research_library', 'fred', 'github', 'etf'])

In [12]:
vs = FREDChromaDocumentLoader(DB_PATH).vectorstore
coll = vs._collection

In [13]:
client = vs._client
print([c.name for c in client.list_collections()])
print("Opened:", coll.name)
print("total docs:", coll.count())

['fred']
Opened: fred
total docs: 220401


## Verify Document Metadata

In [14]:
probe = coll.get(limit=15000)
metadata_list = probe.get("metadatas") if probe else []
metas = [m for m in metadata_list or [] if m]
len(metas)

15000

In [15]:
keys = set().union(*(m.keys() for m in metas)) if metas else set()
for key in sorted(keys):
    print(key)

category_id
category_name
category_path
filename
frequency
last_updated
leaf_name
observation_end
observation_start
popularity
seasonal_adjustment
series_id
series_title
units


## Search Examples 

In [ ]:
vs = FREDChromaDocumentLoader(DB_PATH).vectorstore
retriever = vs.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 8,
        "fetch_k": 60,
    },
)

hits = retriever.invoke("Which Price series track Farm commodities?")
[print(h) for h in hits]

Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


page_content='# Producer Price Index by Commodity: Farm Products: Grains

## Series Information
- **Series ID:** WPS012
- **Frequency:** Monthly
- **Units:** Index 1982=100
- **Seasonal Adjustment:** Seasonally Adjusted
- **Observation Range:** 1967-01-01 – 2025-09-01
- **Last Updated:** 2025-11-25T09:50:10-06:00
- **Popularity:** 19

## Category Information
- **Category ID:** 33528
- **Category Name:** Farm Products
- **Leaf Name:** Farm Products
- **Category Path:** Prices > Producer Price Indexes (PPI) > Commodity Based > Farm Products' metadata={'observation_end': '2025-09-01', 'last_updated': '2025-11-25T09:50:10-06:00', 'series_title': 'Producer Price Index by Commodity: Farm Products: Grains', 'series_id': 'WPS012', 'units': 'Index 1982=100', 'popularity': 19, 'category_name': 'Farm Products', 'frequency': 'Monthly', 'category_path': 'Prices > Producer Price Indexes (PPI) > Commodity Based > Farm Products', 'seasonal_adjustment': 'Seasonally Adjusted', 'leaf_name': 'Farm Product

[None, None, None, None, None, None, None, None]

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


## Metadata Queries

In [17]:
category_counts = Counter(m.get("category_name") for m in metas if "category_name" in m and m.get("category_name"))
for category, count in category_counts.most_common(50):
    print(f"{count:5d}  {category}")

 1000  Section 1. General Statistics of All Banks in the United States
 1000  Shares in Output-side Real GDP at Current Purchasing Power Parities
  950  Price Levels of GDP, Consumption, Government, and Investment
  874  Real GDP, Employment and Population Levels
  677  Economic Policy Uncertainty
  616  Current Price GDP, Capital and Total Factor Productivity
  579  National Accounts-Based Variables
  570  Component Shares of GDP per Capita (Constant Prices)
  570  Component Shares of GDP per Capita (Current Prices)
  570  GDP per Capita (Current Prices)
  570  Real GDP per Capita (Constant Prices)
  501  Exchange Rates and GDP Price Levels
  495  Section 2. Assets and Liabilities of All Member Banks
  488  Income and Employment
  454  Production of Commodities
  385  Construction
  380  Openness
  380  Real GDP per Capita Relative to U.S.
  368  Real GDP per Worker
  335  Prices
  192  Stocks of Commodities
  190  Exchange Rates
  190  Gross Domestic Income, Adjustments for Changes i

## Metadata Key Values

Scan the full collection to find all distinct values for each filterable metadata field.
Use these to validate filter values before passing them to the agent or similarity search.

In [18]:
import statistics

FILTER_KEYS = ["category_name", "frequency", "seasonal_adjustment"]

distinct: dict[str, set] = {k: set() for k in FILTER_KEYS}
for meta in metas:
    for key in FILTER_KEYS:
        val = meta.get(key)
        if val:
            distinct[key].add(val)

for key in FILTER_KEYS:
    values = sorted(distinct[key])
    print(f"\n{key} ({len(values)} distinct values):")
    for v in values:
        print(f"  {v!r}")

pops: list[float] = [float(p) for m in metas if isinstance(p := m.get("popularity"), (int, float))]
if pops:
    pop_max = int(max(pops))
    buckets = [0, 5, 10, 25, 50, 100, pop_max + 1]
    print(f"\npopularity: min={int(min(pops))}  max={pop_max}  mean={statistics.mean(pops):.1f}  median={statistics.median(pops)}")
    for lo, hi in zip(buckets, buckets[1:]):
        count = sum(1 for p in pops if lo <= p < hi)
        print(f"  [{lo:>4} – {hi - 1:>4}): {count}")


category_name (52 distinct values):
  'Component Shares of GDP per Capita (Constant Prices)'
  'Component Shares of GDP per Capita (Current Prices)'
  'Construction'
  'Current Price GDP, Capital and Total Factor Productivity'
  'Daily Federal Funds Rate, 1928-54'
  'Data on the nominal term structure model from Kim and Wright'
  'Distribution of Commodities'
  'Economic Policy Uncertainty'
  'Exchange Rates'
  'Exchange Rates and GDP Price Levels'
  'Financial Status of Business'
  'Foreign Trade'
  'GDP per Capita (Current Prices)'
  'Government and Finance'
  'Gross Domestic Income, Adjustments for Changes in Terms of Trade'
  'Historical Federal Reserve Data'
  'Income and Employment'
  'Interest Rates'
  'Leading Indicators'
  'Money and Banking'
  'National Accounts-Based Variables'
  'New England Textile Industry, 1815-1860'
  'Openness'
  'Population'
  'Price Levels of GDP, Consumption, Government, and Investment'
  'Price Levels, Expenditure Categories and Capital'
  'Prices

In [19]:
# Search within a key's distinct values to validate a specific term
search_key = "category_name"
search_term = "Commodities"

matches = sorted(v for v in distinct[search_key] if search_term.lower() in v.lower())
print(f"{search_key} values containing {search_term!r}:")
for v in matches:
    print(f"  {v!r}")

category_name values containing 'Commodities':
  'Distribution of Commodities'
  'Production of Commodities'
  'Stocks of Commodities'


### Extract Metadata from Request

In [20]:
vs = FREDChromaDocumentLoader(DB_PATH).vectorstore
retriever = vs.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 50,
        "fetch_k": 100,
    },
)

user_request = "What time series are available Commodities in the FRED data?"
hits = retriever.invoke(user_request)
print(len(hits), "hits for query:", user_request)

for i, hit in enumerate(hits, 1):
    metadata = hit.metadata
    print(f"Series ID: {metadata.get('series_id')}")
    print(f"Series Title: {metadata.get('series_title')}")
    print(f"Category:{metadata.get('category_id')}, {metadata.get('category_name')}, {metadata.get('popularity')}")


50 hits for query: What time series are available Commodities in the FRED data?
Series ID: M0501FUSM391NNBR
Series Title: Commercial Stocks of Wheat for United States
Category:33066, Stocks of Commodities, 1
Series ID: M0521AUSM391NNBR
Series Title: Visible Supply of Corn for United States
Category:33066, Stocks of Commodities, 2
Series ID: M0501CUSM391NNBR
Series Title: Visible Supply of Wheat for United States
Category:33066, Stocks of Commodities, 1
Series ID: M0503FUSM596NNBR
Series Title: Visible Supply of Cotton for United States
Category:33066, Stocks of Commodities, 1
Series ID: M0106AUSM234NNBR
Series Title: Book Publication, Editions for United States
Category:33062, Production of Commodities, 1
Series ID: M0521DUSM391NNBR
Series Title: Corn, Commercial Stocks for United States
Category:33066, Stocks of Commodities, 1
Series ID: M0510AUSM027SNBR
Series Title: Manufacturers' Inventories, Finished Goods for United States
Category:33066, Stocks of Commodities, 1
Series ID: A054G

### Series

In [21]:
probe = coll.get(
    where={"category_name": "Farm Products"},
    limit=200
)
metadata_list = probe.get("metadatas") if probe else []
names = [(m["series_id"], m["series_title"]) for m in metadata_list or [] if m]

for name in names:
    print(name)

('WPS01', 'Producer Price Index by Commodity: Farm Products (DISCONTINUED)')
('WPS011', 'Producer Price Index by Commodity: Farm Products: Fruits and Melons, Fresh/Dry Vegetables and Nuts (DISCONTINUED)')
('WPS0111', 'Producer Price Index by Commodity: Farm Products: Fresh Fruits and Melons')
('WPS011101', 'Producer Price Index by Commodity: Farm Products: Citrus Fruits')
('WPS011102', 'Producer Price Index by Commodity: Farm Products: Other Fruits and Berries')
('WPS0113', 'Producer Price Index by Commodity: Farm Products: Fresh and Dry Vegetables (DISCONTINUED)')
('WPS011301', 'Producer Price Index by Commodity: Farm Products: Dry Vegetables')
('WPS011302', 'Producer Price Index by Commodity: Farm Products: Fresh Vegetables, Except Potatoes')
('WPS011303', 'Producer Price Index by Commodity: Farm Products: Sweet Potatoes')
('WPS011306', 'Producer Price Index by Commodity: Farm Products: Potatoes')
('WPS012', 'Producer Price Index by Commodity: Farm Products: Grains')
('WPS0121', 'Pro

In [16]:
paths = [m["category_path"] for m in metadata_list or [] if m]
for path in paths:
    print(path)

Prices > Producer Price Indexes (PPI) > Commodity Based > Farm Products
Prices > Producer Price Indexes (PPI) > Commodity Based > Farm Products
Prices > Producer Price Indexes (PPI) > Commodity Based > Farm Products
Prices > Producer Price Indexes (PPI) > Commodity Based > Farm Products
Prices > Producer Price Indexes (PPI) > Commodity Based > Farm Products
Prices > Producer Price Indexes (PPI) > Commodity Based > Farm Products
Prices > Producer Price Indexes (PPI) > Commodity Based > Farm Products
Prices > Producer Price Indexes (PPI) > Commodity Based > Farm Products
Prices > Producer Price Indexes (PPI) > Commodity Based > Farm Products
Prices > Producer Price Indexes (PPI) > Commodity Based > Farm Products
Prices > Producer Price Indexes (PPI) > Commodity Based > Farm Products
Prices > Producer Price Indexes (PPI) > Commodity Based > Farm Products
Prices > Producer Price Indexes (PPI) > Commodity Based > Farm Products
Prices > Producer Price Indexes (PPI) > Commodity Based > Farm P